In [1]:
import pandas as pd
import os

In [2]:
pd.set_option('display.max_columns', None)

### Files

See **parameter explanation for detail information

In [3]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names

In [4]:
files

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes']

In [5]:
shared_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/R3Data"

### 2. pathology_findings 
<span style="color: blue;">Change idx value **based on EHR file**</span>

In [7]:
idx = 3
file = files[idx-1]
print(file)
extract_cols = pd.read_excel(file_path, sheet_name=file)["Name"].tolist()
print(extract_cols)

pathology_findings
['PATIENT_STUDY_ID', 'BX_ID', 'PATHOLOGY_DATE', 'LESION_CLASS', 'PATHOLOGY_CD', 'HISTROLOGY_GRADE', 'ESTROGEN_RECEPTOR', 'PROGESTERONE_RECEPTOR', 'HER2NEU', 'STAGE_T', 'STAGE_N', 'STAGE_M', 'STAGE_NUM', 'MARGIN_STATUS']


In [8]:
file_names = [
    str(file) + ".txt", 
    str(file) + ".csv",
    str(file) + ".csv"
]
file_names

['pathology_findings.txt', 'pathology_findings.csv', 'pathology_findings.csv']

### Cancer

In [9]:
study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]

In [10]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        merged = pd.merge(
            current_df, 
            all_prior_data, 
            on=list(data_columns), # Merge on all data columns
            how='left', 
            indicator=True
        )
        
        # Rows in the current file that are *only* in the 'left' (current_df) are unique
        unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
        
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [11]:
final_df

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,FINDING_ID,PATHOLOGY_CD,LESION_CLASS,FINDING_SIZE_1,FINDING_SIZE_2,MEASUREMENT_TYPE,HISTROLOGY_GRADE,NODES_REMOVED,NODES_POSITIVE,MARGIN_STATUS,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,NIPPLE_INVOLVED
0,4337721563,482678,01/09/2020,42254322,163533,DCH,MALIGN,NaN,NaN,NaN,G3,NaN,NaN,NaN,P,N,NaN,Ductal carcinoma in situ,NaN,NaN,Stage 0,N
1,4333499919,481991,02/06/2020,741762,164211,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
2,4333872128,481997,02/06/2020,3124344,164215,ID,MALIGN,22.0,NaN,Radiologic,G3,2.0,0.0,NaN,P,P,P,Tumor more than 5 cm in greatest dimension,No regional lymph node metastasis,NaN,Stage 3B,N
3,4333872128,481997,02/06/2020,3124344,164215,DCI,MALIGN,22.0,NaN,Radiologic,G3,2.0,0.0,NaN,P,P,P,Tumor more than 5 cm in greatest dimension,No regional lymph node metastasis,NaN,Stage 3B,N
4,4333872128,481996,02/06/2020,3124344,164216,ID,MALIGN,22.0,NaN,Radiologic,G3,2.0,0.0,NaN,P,P,P,Tumor more than 5 cm in greatest dimension,No regional lymph node metastasis,NaN,Stage 3B,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18391,4335673096,352938,11/28/2022,6719081,192986,FA,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
18392,4333689962,350553,12/14/2022,2142937,194360,ID,MALIGN,19.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,N,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 1,N
18393,4333689962,350553,12/14/2022,2142937,194360,II,MALIGN,19.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,N,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 1,N
18394,4334447886,350012,12/16/2022,1635124,194901,IL,MALIGN,6.0,NaN,Pathologic,G2,NaN,NaN,NaN,P,N,NaN,Tumor more than 0.5 cm but not more than 1 cm ...,"Regional lymph nodes cannot be assessed (e.g.,...",NaN,Stage 1,N


In [12]:
final_df['PATHOLOGY_DATE'] = pd.to_datetime(final_df['PATHOLOGY_DATE'], format='%m/%d/%Y')
final_df['PATHOLOGY_DATE'] = final_df['PATHOLOGY_DATE'].dt.strftime('%Y-%m-%d')

In [13]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', 'BX_ID'], ascending=[True, True, False], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [14]:
unique_df = final_df.copy()

In [15]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [16]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [17]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [18]:
unique_df[unique_df["PATIENT_STUDY_ID"]==4330018595]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,FINDING_ID,PATHOLOGY_CD,LESION_CLASS,FINDING_SIZE_1,FINDING_SIZE_2,MEASUREMENT_TYPE,HISTROLOGY_GRADE,NODES_REMOVED,NODES_POSITIVE,MARGIN_STATUS,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,NIPPLE_INVOLVED,duplicate_count
0,4330018595,487690,2020-06-05,9654707,168529,ID,MALIGN,30.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,No distant metastasis,Stage 1,N,1
1,4330018595,487690,2020-06-05,9654707,168529,DCI,MALIGN,30.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,No distant metastasis,Stage 1,N,1
2,4330018595,475789,2020-07-28,9654707,170446,II,MALIGN,30.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,No distant metastasis,Stage 1,N,1


In [19]:
unique_df['duplicate_count'].unique()

array([1, 2, 3], dtype=int64)

In [20]:
unique_df[unique_df['duplicate_count']!=1]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,FINDING_ID,PATHOLOGY_CD,LESION_CLASS,FINDING_SIZE_1,FINDING_SIZE_2,MEASUREMENT_TYPE,HISTROLOGY_GRADE,NODES_REMOVED,NODES_POSITIVE,MARGIN_STATUS,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,NIPPLE_INVOLVED,duplicate_count
170,4330375231,487557,2019-11-14,45028177,167571,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
171,4330375231,487557,2019-11-14,45028177,167572,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
295,4330460996,427070,2018-11-05,43212929,130325,RS,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
296,4330460996,427070,2018-11-05,43212929,130326,FF,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
297,4330460996,427070,2018-11-05,43212929,130325,SA,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
2512,4333186149,470719,2021-02-22,2712006,175346,LS,HIGHRISK,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
2513,4333186149,470719,2021-02-22,2712006,175347,ALH,HIGHRISK,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
4283,4333370935,412539,2019-03-04,1074757,133887,DHU,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
4284,4333370935,412539,2019-03-04,1074757,133888,DHU,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2
8408,4333835305,486771,2020-02-13,2564015,169455,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2


In [21]:
unique_df = final_df.drop_duplicates(subset=final_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [22]:
unique_df

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,FINDING_ID,PATHOLOGY_CD,LESION_CLASS,FINDING_SIZE_1,FINDING_SIZE_2,MEASUREMENT_TYPE,HISTROLOGY_GRADE,NODES_REMOVED,NODES_POSITIVE,MARGIN_STATUS,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,NIPPLE_INVOLVED
0,4330018595,487690,2020-06-05,9654707,168529,ID,MALIGN,30.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,No distant metastasis,Stage 1,N
1,4330018595,487690,2020-06-05,9654707,168529,DCI,MALIGN,30.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,No distant metastasis,Stage 1,N
2,4330018595,475789,2020-07-28,9654707,170446,II,MALIGN,30.0,NaN,Pathologic,G2,2.0,0.0,NaN,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,No distant metastasis,Stage 1,N
3,4330029102,477464,2021-05-10,7939240,177674,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
4,4330044371,467808,2022-05-23,2438374,188141,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18344,4339933706,449749,2017-07-25,43137037,108450,IP,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
18345,4339943307,410232,2019-06-25,43135668,136168,FA,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
18346,4339945522,353888,2022-10-25,9099455,192141,ADH,HIGHRISK,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
18347,4339945522,350178,2022-12-15,9099455,194844,ADH,HIGHRISK,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N


In [23]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)

### Control

In [79]:
file_names = [
    str(file) + "_controls.txt", 
    str(file) + "_controls.csv",
]
file_names

['pathology_findings_controls.txt', 'pathology_findings_controls.csv']

In [80]:
study = "Control"
folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]

In [81]:
final_unique_dfs = [] # List to hold the de-duplicated data from each file

# 2. Process each file
for i, file_name in enumerate(file_names):
    file_extension = os.path.splitext(file_name)[1].lower()
    file_path = os.path.join(shared_path, study, folder[i], file_name)
    
    if file_extension == '.csv':
        try:
            current_df = pd.read_csv(file_path, encoding='latin-1')
        except:
            print('file not exist in:', folder[i])
    elif file_extension == '.txt':
        try:
            current_df = pd.read_csv(file_path, sep='|')
        except:
            print('file not exist in:', folder[i])
    else:
        continue

    # --- KEY LOGIC FOR INTER-FILE DEDUPLICATION ---
    
    # Concatenate the current file's data with all previously processed data
    # (Do NOT use ignore_index=True here, we need separate indices for now)
    
    if final_unique_dfs:
        # Create a temporary DataFrame of ALL data processed so far
        all_prior_data = pd.concat(final_unique_dfs)
        
        # Check the current_df against all prior data. 
        # We only look at the columns that contain the actual data (excluding the index).
        data_columns = current_df.columns
        
        # Identify rows in the current file that are NOT duplicates of PRIOR files
        # The 'indicator=True' is used to identify the source of the merge
        try:
            merged = pd.merge(
                current_df, 
                all_prior_data, 
                on=list(data_columns), # Merge on all data columns
                how='left', 
                indicator=True
            )

            
            # Rows in the current file that are *only* in the 'left' (current_df) are unique
            unique_to_current_file = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')

        except:
            merged = pd.concat((all_prior_data, current_df), ignore_index=True)
            unique_to_current_file = merged
            
        # Append the unique data (which includes any intra-file duplicates)
        final_unique_dfs.append(unique_to_current_file)
    else:
        # The first file is added as is (it has no prior files to check against)
        final_unique_dfs.append(current_df)


# 3. Final Concatenation
if final_unique_dfs:
    # Concatenate the final list of DataFrames to produce the result
    final_df = pd.concat(final_unique_dfs, ignore_index=True)

    print("✅ Successfully merged and removed ONLY inter-file duplicates.")
    print("\nFinal DataFrame head (includes intra-file duplicates):")
else:
    print("❌ No valid files were read. The final DataFrame is empty.")

✅ Successfully merged and removed ONLY inter-file duplicates.

Final DataFrame head (includes intra-file duplicates):


In [82]:
final_df

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,FINDING_ID,PATHOLOGY_CD,LESION_CLASS,FINDING_SIZE_1,FINDING_SIZE_2,MEASUREMENT_TYPE,HISTROLOGY_GRADE,NODES_REMOVED,NODES_POSITIVE,MARGIN_STATUS,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,NIPPLE_INVOLVED
0,4333062366,475428,08/06/2020,2421647,169708,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
1,4333062366,471969,12/04/2020,2421647,174232,PA,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
2,4333373719,471830,10/29/2020,8400344,174261,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
3,4333373719,471839,10/29/2020,8400344,174262,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
4,4334928638,426146,01/02/2019,44319710,131172,AM,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
721,4333229846,351240,10/03/2022,9083419,193670,ID,MALIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
722,4333017529,467603,04/06/2022,9315511,188333,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
723,4335929637,466349,04/18/2022,8206626,188598,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
724,4333007393,466178,04/11/2022,3010324,188865,II,MALIGN,30.0,NaN,Pathologic,G3,NaN,NaN,NaN,P,P,N,Tumor more than 2 cm but not more than 5 cm in...,No regional lymph node metastasis,No distant metastasis,Stage 2A,N


In [83]:
final_df['PATHOLOGY_DATE'] = pd.to_datetime(final_df['PATHOLOGY_DATE'], format='%m/%d/%Y')
final_df['PATHOLOGY_DATE'] = final_df['PATHOLOGY_DATE'].dt.strftime('%Y-%m-%d')

In [84]:
final_df.sort_values(by=['PATIENT_STUDY_ID', 'PATHOLOGY_DATE', "BX_ID"], ascending=[True, True, False], inplace=True, ignore_index=True)

#### <span style="color: blue;">Remove duplicated entries, add a column marked how many duplicates </span> 

**Before drop duplicate = <span style="color: blue;">final_df</span> , after drop duplicate = <span style="color: blue;">unique_df</span>**

In [85]:
unique_df = final_df.copy()

In [86]:
unique_df['duplicate_count'] = unique_df.groupby(unique_df.columns.tolist(), dropna=False).transform('size')

In [87]:
unique_df = unique_df.drop_duplicates(subset=unique_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [88]:
unique_extract_cols = extract_cols + ['duplicate_count']

In [89]:
unique_df['duplicate_count'].unique()

array([1, 3, 2])

In [91]:
unique_df[unique_df['duplicate_count']!=1]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,PATIENT_ID,FINDING_ID,PATHOLOGY_CD,LESION_CLASS,FINDING_SIZE_1,FINDING_SIZE_2,MEASUREMENT_TYPE,HISTROLOGY_GRADE,NODES_REMOVED,NODES_POSITIVE,MARGIN_STATUS,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,NIPPLE_INVOLVED,duplicate_count
27,4333008024,410317,2019-06-17,1174803,136092,AD,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
53,4333062366,475428,2020-08-06,2421647,169708,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
54,4333062366,471969,2020-12-04,2421647,174232,PA,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
79,4333101704,355101,2022-07-15,1192796,189825,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
145,4333373719,471839,2020-10-29,8400344,174262,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
146,4333373719,471830,2020-10-29,8400344,174261,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
161,4333488305,417825,2019-11-07,8194055,139586,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
165,4333514173,478396,2021-03-19,9687278,176762,AN,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
168,4333571001,427540,2018-11-12,8262906,129745,BC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3
181,4333624870,469223,2022-02-21,9760831,185736,FC,BENIGN,NaN,NaN,NaN,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,3


In [92]:
unique_df = final_df.drop_duplicates(subset=final_df.columns.tolist(), keep = 'first').reset_index(drop = True)

In [93]:
unique_df[extract_cols]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,PATHOLOGY_CD,HISTROLOGY_GRADE,ESTROGEN_RECEPTOR,PROGESTERONE_RECEPTOR,HER2NEU,STAGE_T,STAGE_N,STAGE_M,STAGE_NUM,MARGIN_STATUS
0,4330189702,418851,2019-04-26,MALIGN,ID,G2,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,Metastasis in 1 to 3 axillary lymph nodes,Distant metastasis cannot be assessed,Stage 2A,NaN
1,4330311096,450823,2016-11-10,MALIGN,ID,G2,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 1,NaN
2,4330311096,450823,2016-11-10,MALIGN,II,G2,P,P,N,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 1,NaN
3,4330311096,459526,2016-11-28,MALIGN,II,G2,P,P,NaN,Tumor more than 1 cm but not more than 2 cm in...,No regional lymph node metastasis,Distant metastasis cannot be assessed,Stage 1,NaN
4,4330329490,437485,2017-10-25,BENIGN,IP,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
617,4337609101,406901,2015-08-20,BENIGN,FF,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
618,4337636503,406140,2015-03-09,BENIGN,FNS,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
619,4337689949,420192,2018-08-02,BENIGN,FC,GX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
620,4337793359,414656,2019-01-18,MALIGN,ID,G2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
# output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
# final_df[extract_cols].to_excel(output_file, index=False)

output_file = os.path.join("../Data/", study, "Cleaned", file + ".xlsx")
unique_df[extract_cols].to_excel(output_file, index=False)